# 44 — Greedy singles λ = 0 vs 0.4 on the bf16 validated pool

Removes the pool/precision confound in the §5.6 single-vs-multi λ
comparison. The report's singles λ result (nb31) is on the *legacy* 27
concepts at AWQ; the multi λ result (nb40/41) is on the *bf16-validated*
pool. This notebook measures single-concept steering at λ ∈ {0, 0.4} on
the **same bf16-validated pool** the multi results use, through the same
unified greedy loop (the canonical wrapper has no fluency dial), so
singles and multi are finally on one footing.

40 pool concepts × 2 λ × 10 prompts. Question: is λ "better than free" on
singles (steering AND fluency improve) even on validated guides — while it
is a plain trade-off on composition?


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS, K, STEPS = 3, 4, 24
LAMBDAS = [0.0, 0.4]
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
CONCEPTS = [r["synset"] for r in POOL["pool"]]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/44_singles_lambda_bf16.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"{len(CONCEPTS)} pool concepts x {len(LAMBDAS)} lambda")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
concept = {n: fur.get_concept_cached(n, MIN_LEMMAS, MAX_TOKENS) for n in CONCEPTS}
print(f"{len(concept)} concepts built")

for name in CONCEPTS:
    c = concept[name]
    for lam in LAMBDAS:
        torch.cuda.empty_cache()
        # unified greedy loop, single guide, fluency dial (canonical wrapper has none)
        texts, probe = fur.generate_with_topk_multi_guide(
            prompts, guides=[c], k=K, steps=STEPS, fluency_weight=lam
        )
        harness.evaluate(
            prompts,
            texts,
            [name],
            config={
                "model": MODEL_KEY,
                "method": "single",
                "lam": lam,
                "concept": name,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts={name: c},
            angles=True,
        )
    print(f"{name}: done")

In [ ]:
rows = []
for line in Path(LOG_PATH).read_text().strip().split("\n"):
    r = json.loads(line)
    c = r["config"]
    name = c["concept"]
    rows.append(
        {
            "lam": c["lam"],
            "concept": name,
            "delta": r["expression"][name]["delta"],
            "d_cos": r["angles"][name]["delta"]["token_cos"],
            "ppl_ratio": r["ppl_ratio"],
        }
    )
df = pd.DataFrame(rows)
summary = (
    df.groupby("lam")
    .agg(
        delta=("delta", "mean"),
        positive=("delta", lambda s: (s > 0).mean()),
        d_cos=("d_cos", "mean"),
        positive_cos=("d_cos", lambda s: (s > 0).mean()),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("delta", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/44_singles_lambda_summary.csv", index=False)
print(summary.round(4).to_string(index=False))

s0 = summary[summary.lam == 0.0].iloc[0]
s4 = summary[summary.lam == 0.4].iloc[0]
print()
print("SINGLES (bf16 validated), lambda 0 -> 0.4:")
print(
    f"  delta   {s0['delta']:.3f} -> {s4['delta']:.3f}  ({s4['delta']/s0['delta']:.2f}x)"
)
print(
    f"  d_cos   {s0['d_cos']:.4f} -> {s4['d_cos']:.4f}  ({s4['d_cos']/s0['d_cos']:.2f}x)"
)
print(f"  positive {s0['positive']:.0%} -> {s4['positive']:.0%}")
print(f"  ppl     {s0['ppl_ratio']:.1f} -> {s4['ppl_ratio']:.1f}")
verdict = (
    "BETTER THAN FREE (steering up)"
    if s4["d_cos"] > s0["d_cos"]
    else "TRADE-OFF (steering down)"
)
print(f"  -> singles verdict: {verdict}")
print("  (compare multi blend nb40: d_cos DOWN ~12-20%, ppl down -> trade-off)")

In [ ]:
labels = ["lam=0", "lam=0.4"]
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, (col, lab) in zip(
    ax,
    (
        ("d_cos", "singles token-cosine Δ"),
        ("positive_cos", "positive rate (cos)"),
        ("ppl_ratio", "perplexity ratio"),
    ),
):
    a.bar(
        labels,
        [summary[summary.lam == lm][col].iloc[0] for lm in (0.0, 0.4)],
        color=["tab:gray", "tab:blue"],
    )
    a.set_ylabel(lab)
    a.axhline(0, color="gray", lw=0.6)
ax[2].axhline(2.5, color="red", ls=":", lw=0.8)
plt.suptitle("Greedy singles: fluency dial on the bf16 validated pool")
plt.tight_layout()
plt.savefig("resources/44_singles_lambda.png", dpi=150)
plt.show()